# 00 - System Checkpoint

This notebook answers: **what exists, what passes, and what evidence can I inspect before touching the ingestion system?**

It is the first story notebook. It checks the public CLI surface, source registry, database schema model, generated artifacts, notebook inventory, and test suite. It does not run the agent pipeline, does not call live provider websites, and does not mutate data.


## 1. Setup

Configure paths, logging, display helpers, and public imports. The imports intentionally use project entry points and documented schema surfaces rather than duplicating production logic.


In [ ]:
print("[STAGE] setup")

import argparse
import json
import logging
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("story_notebook.system_checkpoint")

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mutual_fund_ingestion import load_registry
from mutual_fund_ingestion.agent.db import Base
from mutual_fund_ingestion.cli import build_parser

NOTEBOOK_DIR = ROOT / "notebooks" / "mutual_fund_ingestion"
DOCS_DIR = ROOT / "docs"
CONFIG_PATH = ROOT / "configs" / "amc_sources.yaml"


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def run_command(command: list[str], timeout: int = 180) -> subprocess.CompletedProcess[str]:
    logger.info("run_command | command=%s timeout_s=%s", " ".join(command), timeout)
    return subprocess.run(
        command,
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=timeout,
        check=False,
    )


def show_table(rows: list[dict], columns: list[str] | None = None) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if columns:
        df = df[columns]
    display(df)
    return df

print(f"  root: {ROOT}")
print(f"  notebook_dir: {NOTEBOOK_DIR}")
assert (ROOT / "AGENTS.md").exists(), "project root must contain AGENTS.md"
assert NOTEBOOK_DIR.exists(), "notebook directory must exist"
print("  setup complete")


## 2. Repository Context

This stage locates the active docs and planning files that define the notebook story. It confirms this notebook is running against the expected repository layout.


In [ ]:
print("[STAGE] repository context")

context_paths = [
    ROOT / "AGENTS.md",
    DOCS_DIR / "README.md",
    DOCS_DIR / "01_status" / "MASTER_STATE.md",
    DOCS_DIR / "02_architecture" / "codebase_map.md",
    DOCS_DIR / "02_architecture" / "database_schema.md",
    DOCS_DIR / "06_plans" / "active" / "STORY_NOTEBOOK_SERIES_PLAN.md",
]

context_rows = []
for item in context_paths:
    context_rows.append({
        "path": rel(item),
        "exists": item.exists(),
        "size_bytes": item.stat().st_size if item.exists() else None,
    })

context_df = show_table(context_rows)
missing_context = [row["path"] for row in context_rows if not row["exists"]]
print(f"  context files checked: {len(context_rows)}")
print(f"  missing context files: {missing_context or 'none'}")

assert not missing_context, f"missing required context files: {missing_context}"


## 3. Public CLI Surface

The CLI is the user-facing entry point for source bootstrapping, provider profiling, database initialization, agent runs, inspection, and retry handling. This stage verifies the expected commands are present.


In [ ]:
print("[STAGE] public CLI surface")

parser = build_parser()
subparser_action = next(
    action for action in parser._actions if isinstance(action, argparse._SubParsersAction)
)
commands = sorted(subparser_action.choices)
required_commands = {
    "bootstrap-sources",
    "profile-providers",
    "phase-1",
    "init-db",
    "run-agent",
    "inspect-run",
    "retry-failed",
}

cli_rows = [{"command": command, "required": command in required_commands} for command in commands]
cli_df = show_table(cli_rows)
missing_commands = sorted(required_commands - set(commands))

print(f"  commands found: {commands}")
print(f"  missing required commands: {missing_commands or 'none'}")

assert required_commands.issubset(set(commands)), f"missing CLI commands: {missing_commands}"
assert "profile-sites" in commands, "compatibility alias profile-sites should remain available"


## 4. Database Schema Inventory

The system story eventually reaches database persistence. This stage inspects the SQLAlchemy model registry without connecting to a live database.


In [ ]:
print("[STAGE] database schema inventory")

table_names = sorted(Base.metadata.tables.keys())
expected_core_tables = {
    "ingestion_runs",
    "task_urls",
    "source_pages",
    "discovered_links",
    "dataset_candidates",
    "raw_artifacts",
    "staging_rows",
    "validation_results",
    "quarantine_rows",
    "amcs",
    "schemes",
    "nav_history",
    "portfolio_snapshots",
    "portfolio_holdings",
}

db_rows = []
for table_name in table_names:
    table = Base.metadata.tables[table_name]
    db_rows.append({
        "table": table_name,
        "columns": len(table.columns),
        "core_story_table": table_name in expected_core_tables,
    })

db_df = show_table(db_rows)
missing_tables = sorted(expected_core_tables - set(table_names))

print(f"  table count: {len(table_names)}")
print(f"  missing core story tables: {missing_tables or 'none'}")

assert len(table_names) == 17, f"expected 17 DB tables, found {len(table_names)}"
assert expected_core_tables.issubset(set(table_names)), f"missing core DB tables: {missing_tables}"


## 5. Source Registry Snapshot

The provider-first story starts with the source registry. This stage loads the registry through the public package entry point and summarizes roles, enabled status, and provider/reference counts.


In [ ]:
print("[STAGE] source registry snapshot")

registry_entries = load_registry(CONFIG_PATH)
registry_rows = []
for entry in registry_entries:
    item = entry.to_dict() if hasattr(entry, "to_dict") else dict(entry.__dict__)
    registry_rows.append({
        "source_name": item.get("source_name"),
        "amc_name": item.get("amc_name"),
        "source_role": item.get("source_role"),
        "source_type": item.get("source_type"),
        "enabled": item.get("enabled"),
        "priority": item.get("priority"),
        "seed_url_present": bool(item.get("seed_url")),
    })

registry_df = pd.DataFrame(registry_rows)
summary_df = registry_df.groupby(["source_role", "enabled"], dropna=False).size().reset_index(name="count")
display(summary_df)
display(registry_df.head(12))

primary_enabled = registry_df[(registry_df["source_role"] == "primary_provider") & (registry_df["enabled"] == True)]
reference_entries = registry_df[registry_df["source_role"] == "reference_index"]

print(f"  registry entries: {len(registry_df)}")
print(f"  enabled primary providers: {len(primary_enabled)}")
print(f"  reference index entries: {len(reference_entries)}")

assert len(registry_entries) > 0, "registry must not be empty"
assert len(primary_enabled) > 0, "registry must contain enabled primary providers"
assert len(reference_entries) >= 2, "registry should include AMFI/SEBI reference entries"


## 6. Artifact Inventory

This stage checks the artifact surface the story notebooks will inspect. Missing optional artifacts are shown as evidence gaps, not hidden.


In [ ]:
print("[STAGE] artifact inventory")

artifact_paths = [
    ("source registry latest", ROOT / "data" / "raw" / "mutual_funds" / "source_registry" / "source_registry.latest.json", True),
    ("source registry candidates", ROOT / "data" / "raw" / "mutual_funds" / "source_registry" / "source_registry_candidates.jsonl", False),
    ("provider profiles latest", ROOT / "data" / "raw" / "mutual_funds" / "provider_profiles" / "provider_profiles.latest.json", False),
    ("provider profile summary", ROOT / "data" / "reports" / "mutual_funds" / "provider_profile_summary.csv", False),
    ("provider profile report", ROOT / "data" / "reports" / "mutual_funds" / "provider_profile_report.html", False),
    ("runtime raw dir", ROOT / "data" / "raw" / "mutual_funds" / "runtime", False),
    ("runtime temp dir", ROOT / "data" / "tmp" / "mutual_funds" / "runtime", False),
]

artifact_rows = []
for label, item, required in artifact_paths:
    artifact_rows.append({
        "artifact": label,
        "path": rel(item),
        "exists": item.exists(),
        "required_for_00": required,
        "size_bytes": item.stat().st_size if item.is_file() else None,
    })

artifact_df = show_table(artifact_rows)
missing_required_artifacts = [row["path"] for row in artifact_rows if row["required_for_00"] and not row["exists"]]
optional_missing = [row["path"] for row in artifact_rows if not row["required_for_00"] and not row["exists"]]

print(f"  missing required artifacts: {missing_required_artifacts or 'none'}")
print(f"  optional missing artifacts: {optional_missing or 'none'}")

assert not missing_required_artifacts, f"missing required artifacts: {missing_required_artifacts}"
assert any(row["exists"] for row in artifact_rows), "at least one story artifact should exist"


## 7. Notebook Inventory

The current notebook layer contains both retained story notebooks and compatibility notebooks that will be rewritten or replaced later. This stage records the current inventory without deleting anything.


In [ ]:
print("[STAGE] notebook inventory")

expected_notebooks = {
    "00_system_checkpoint.ipynb": "keep/rewrite as system dashboard",
    "01_phase_1_provider_profiling_review.ipynb": "stale index; later pointer/archive decision",
    "01a_phase_1_source_registry_review.ipynb": "rewrite as source registry story",
    "01b_phase_1_provider_profiling_review.ipynb": "rewrite as provider profile story",
    "02_agent_pipeline_inspection.ipynb": "keep/rewrite as canonical agent story",
    "02_task_url_ingestion_agent_inspection.ipynb": "duplicate; later pointer/archive decision",
    "03_phase2_discovery_review.ipynb": "rewrite as discovery/candidate story",
}

actual_notebooks = sorted(path.name for path in NOTEBOOK_DIR.glob("*.ipynb"))
notebook_rows = []
for name in sorted(set(expected_notebooks) | set(actual_notebooks)):
    item = NOTEBOOK_DIR / name
    notebook_rows.append({
        "notebook": name,
        "exists": item.exists(),
        "planned_status": expected_notebooks.get(name, "unexpected extra notebook"),
        "size_bytes": item.stat().st_size if item.exists() else None,
    })

notebook_df = show_table(notebook_rows)
missing_notebooks = sorted(set(expected_notebooks) - set(actual_notebooks))

print(f"  notebooks found: {actual_notebooks}")
print(f"  missing expected notebooks: {missing_notebooks or 'none'}")

assert not missing_notebooks, f"missing notebooks: {missing_notebooks}"
assert len(actual_notebooks) >= 7, "expected at least seven current notebooks before rewrite series"


## 8. Test Suite Status

This stage runs the project test command so the notebook captures the same pass/fail surface as the status docs. It is the only intentionally slow stage in this notebook.


In [ ]:
print("[STAGE] test suite status")

pytest_cmd = ["./financial_env/bin/python", "-m", "pytest", "tests/", "-q", "--tb=no"]
pytest_result = run_command(pytest_cmd, timeout=240)
combined_output = "\n".join(part for part in [pytest_result.stdout, pytest_result.stderr] if part)
summary_lines = [line for line in combined_output.splitlines() if "passed" in line or "failed" in line or "skipped" in line]
summary = summary_lines[-1] if summary_lines else "no summary found"
summary_match = re.search(r"(\d+) passed(?:, (\d+) skipped)?", summary)
passed_count = int(summary_match.group(1)) if summary_match else None
skipped_count = int(summary_match.group(2) or 0) if summary_match else None

test_df = show_table([
    {
        "command": " ".join(pytest_cmd),
        "returncode": pytest_result.returncode,
        "summary": summary,
        "passed": passed_count,
        "skipped": skipped_count,
    }
])

print("  pytest tail:")
for line in combined_output.splitlines()[-8:]:
    print(f"    {line}")

assert pytest_result.returncode == 0, f"pytest failed with return code {pytest_result.returncode}"
assert passed_count is not None and passed_count >= 145, f"expected at least 145 passing tests, saw {summary}"


## 9. Failure and Debug Surface

A useful checkpoint should show how failures are represented. This stage checks a deliberately missing artifact path and reports it as an inspectable gap rather than hiding it.


In [ ]:
print("[STAGE] failure/debug surface")

missing_path = ROOT / "data" / "raw" / "mutual_funds" / "runtime" / "missing_checkpoint_artifact.example"
try:
    missing_path.read_bytes()
except FileNotFoundError as exc:
    debug_case = {
        "case": "missing optional raw artifact",
        "path": rel(missing_path),
        "error_type": type(exc).__name__,
        "message": str(exc),
        "next_step": "Run a bounded raw-artifact notebook task when raw-download evidence is needed.",
    }
else:
    debug_case = {
        "case": "missing optional raw artifact",
        "path": rel(missing_path),
        "error_type": None,
        "message": "Unexpectedly exists",
        "next_step": "Inspect why the placeholder path exists.",
    }

debug_df = show_table([debug_case])
print(f"  debug case: {debug_case['error_type']} at {debug_case['path']}")

assert debug_case["error_type"] == "FileNotFoundError", "missing artifact debug case should be explicit"


## 10. What This Proves / What It Does Not Prove

### What this proves

- The repository has the expected project context files.
- The public CLI exposes the commands needed for the story notebook series.
- The SQLAlchemy schema still exposes 17 tables.
- The source registry loads through the public package entry point.
- Required story-notebook artifacts and current notebooks are discoverable.
- The full test suite passes at the current baseline when this notebook is executed.

### What this does not prove

- It does not prove live provider websites are reachable today.
- It does not run `run-agent` or download a raw document.
- It does not prove parsing, validation, quarantine, or canonical upserts on fresh live data.
- It does not implement analytics or investment recommendations.

Those behaviors belong to later story notebooks in the series.
